# 📝 청킹 전략·RAPTOR 과제 LV2(응용)

Neo4j 옛 릴리스 노트 5편(영문)과 질문 30개로 검색 전략을 비교합니다. 7번에서 GPT 요약을 최대 3번 호출합니다. 6·7번은 3·4번에서 만든 객체를 이어 씁니다.

맨 위 준비 셀을 차례로 실행한 뒤, 문항마다 답안 셀을 채우고 자가채점 셀로 확인하세요.

In [ ]:
# 파일을 읽고 원문·출처를 담는 공통 도구입니다. 전략별 도구는 해당 문항에서 가져옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from langchain_core.documents import Document

In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
import sys

material_dir = Path(".")
# 학생용·정답용 모두 실습자료 폴더의 util.py를 가져옵니다.
sys.path.insert(0, str(material_dir.resolve()))

data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

In [ ]:
# 환경변수와 모델 연결을 준비하고, 같은 임베딩을 LlamaIndex 파서에도 연결합니다.
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from llama_index.embeddings.langchain import LangchainEmbedding

# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(".env에 OPENAI_API_KEY를 설정한 뒤 이 셀부터 다시 실행하세요.")

# 이 셀은 모델 설정만 준비합니다. GPT 요청은 뒤의 체인 invoke에서 발생합니다.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,  # 요약·답변 요청에 OpenAI Responses API를 사용합니다.
)

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,  # 문장·청크 하나를 768개 숫자로 표현합니다.
    check_embedding_ctx_length=False,  # 자동 길이 검사와 분할을 끄고 준비한 짧은 청크를 보냅니다.
)
# LlamaIndex 파서에는 같은 임베딩을 연결해 씁니다(새 모델이 아닙니다).
index_embedding = LangchainEmbedding(embedding_model)
print("모델 연결 설정 완료. 임베딩은 본문·질문을 벡터로 바꿀 때 요청합니다.")

In [ ]:
def make_documents(records):
    """원문 기록마다 본문과 출처(ID·제목·위치)를 담은 LangChain Document를 만듭니다."""
    return [
        Document(
            page_content=record["text"],
            # 분할한 뒤에도 source_id로 청킹 전 본문을 다시 찾습니다.
            metadata={
                "source_id": record["doc_id"],
                "title": record["title"],
                "url": record["url"],
                # 원문 전체 구간을 먼저 기록하고, 분할 뒤에는 각 청크의 구간으로 갱신합니다.
                "start_index": 0,
                "end_index": len(record["text"]),
            },
        )
        for record in records
    ]

In [ ]:
# 원문과 질문을 읽습니다. 1~5번은 첫 질문(question_text) 하나로 연습합니다.
records = read_json("lv2_docs.json")
documents = make_documents(records)
# 파서 결과의 source_id로 원문을 다시 찾을 때 씁니다.
records_by_id = {record["doc_id"]: record for record in records}

questions = read_json("lv2_questions.json")
question_text = questions[0]["question"]

print(
    "원문 수:", len(records),
    "/ 질문 수:", len(questions),
    "/ 첫 질문:", question_text,
)

In [ ]:
# [제공 코드]
# 시작 위치와 청크 길이로 끝 위치를 기록하는 지원 함수입니다.
from util import set_end_offsets

In [ ]:
# [제공 코드]
# LangChain 문서를 LlamaIndex 파서의 입력 형식으로 바꿀 때 씁니다.
from llama_index.core import Document as IndexDocument

def to_index_documents(documents):
    """LangChain Document를 LlamaIndex 파서가 읽는 Document로 바꿉니다."""
    # 본문·metadata를 유지하면서 파서가 사용하는 Document 형식으로 바꿉니다.
    result = [IndexDocument.from_langchain_format(doc) for doc in documents]
    for doc in result:
        # 분할 노드의 ref_doc_id로 청킹 전 원문을 찾을 수 있도록 ID를 맞춥니다.
        doc.id_ = doc.metadata["source_id"]
        # metadata는 보관하되 모델 입력에서 빼서 긴 URL 등이 분할·의미 비교에 섞이지 않게 합니다.
        doc.excluded_embed_metadata_keys = list(doc.metadata)
        doc.excluded_llm_metadata_keys = list(doc.metadata)

    return result

In [ ]:
# [제공 코드]
# 파서 결과를 Document로 바꾸고 원문 위치를 연결하는 지원 함수입니다.
from util import nodes_to_documents, restore_positions

청크의 원문 위치를 연결하는 보조 함수는 [util.py](util.py)에 있습니다. 교안·과제와 같은 폴더에 두고, 아래 표의 입력과 결과를 확인해 사용합니다.

| 제공 이름 | 내용 |
|---|---|
| `records`, `records_by_id` | 원문 5편, `doc_id → 원문 딕셔너리` |
| `documents` | `Document` 리스트. metadata에 `source_id`·`start_index`·`end_index`(끝 위치는 포함하지 않음) |
| `questions`, `question_text` | 질문 30개(`question_id`, `question`, `evidence`), 첫 질문 문자열 |
| `embedding_model`, `index_embedding`, `llm` | OpenAI 임베딩, 그것을 LlamaIndex 파서에 연결한 객체, GPT 모델 |
| `set_end_offsets(chunks)` | Fixed 분할 결과에 끝 위치를 채웁니다 |
| `to_index_documents(documents)` | 파서 입력으로 바꿉니다 |
| `nodes_to_documents(nodes, all_nodes=None, records_by_id=None)` | 평면 파서는 첫 인자만 씁니다. 계층은 전체 `auto_nodes`와 청킹 전 `records_by_id`도 전달합니다 |
| `restore_positions(docs, records_by_id, sequential=False)` | 초기 Semantic·Window 전체 결과는 `sequential=True`, 검색 후 결과는 기본값으로 위치를 붙입니다 |
| `evidence_recall(contexts, evidence)` | 반환 문서들이 근거 구간을 모두 덮은 비율(6번 앞에서 제공) |
| `summarize_groups(groups, level, summary_chain)` | 군집마다 GPT 요약 한 번 + `child_ids`·`leaf_ids` 연결(7번 앞에서 제공) |

## 1. 분할 결과를 원문 구간과 연결하기

**배경**: 청크의 출처와 위치로 원문을 다시 찾습니다.

**요구사항**:

- **`fixed_parts`** (`Document` 리스트): `documents` 전체를 문자 단위 Fixed 180자·겹침 30으로 나누고 끝 위치까지 기록한 결과.
- **`source_matches`** (불리언 리스트): 청크 순서대로 `page_content`가 같은 출처 원문의 `text[start_index:end_index]`와 같은지.

**확인 기준**: 청크 15개, 모두 True. `release0`의 시작 위치는 `[0, 150]`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 분할한 뒤 청크 metadata의 출처와 위치로 원문을 잘라 비교합니다.

세부구현:
1. LV1 1번과 같은 문자 단위 Fixed 설정으로 나누고 set_end_offsets를 적용합니다.
2. 청크마다 source_id로 원문을 찾아 두 위치로 슬라이싱한 뒤 page_content와 비교합니다.
```

</details>

In [ ]:
# [제공 코드]
# 고정 길이 분할에 사용할 분할기입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(fixed_parts) == 15 and all(len(doc.page_content) <= 180 for doc in fixed_parts), (
    f"청크가 {len(fixed_parts)}개입니다. documents 전체를 180자·겹침 30으로 나누면 15개입니다."
)

starts = [doc.metadata["start_index"] for doc in fixed_parts if doc.metadata["source_id"] == "release0"]

assert starts == [0, 150], (
    f"release0 시작 위치가 {starts}입니다. 구분자를 빈 문자열 하나로, add_start_index=True로 두세요."
)
assert all(doc.metadata["end_index"] == doc.metadata["start_index"] + len(doc.page_content) for doc in fixed_parts), (
    "끝 위치가 갱신되지 않았습니다. set_end_offsets를 적용하세요."
)
assert source_matches == [
    doc.page_content
    == records_by_id[doc.metadata["source_id"]]["text"][doc.metadata["start_index"] : doc.metadata["end_index"]]
    for doc in fixed_parts
], "source_matches는 청크마다 실제로 비교한 결과를 청크 순서대로 담습니다."
assert all(source_matches), "False가 있습니다. strip_whitespace=False와 set_end_offsets를 확인하세요."

## 2. Semantic 파서의 백분위 비교

**배경**: 거리 기준을 정하는 백분위만 바꿔 경계 수를 비교합니다.

**요구사항**:

- **`percentile50_chunks`** (`Document` 리스트): `documents`를 `SemanticSplitterNodeParser`(`buffer_size=1`, 50백분위, `index_embedding`)로 나누고 원문 위치를 붙인 결과.
- **`percentile80_chunks`** (`Document` 리스트): 같은 설정에서 백분위만 80.
- **위치 연결**: 각 초기 파서 전체 결과를 `nodes_to_documents`로 바꾸고 `restore_positions`에 `sequential=True`로 전달하세요. 이전 청크 끝 다음부터 원문을 찾아 반복 문구를 구별합니다.
- 두 결과의 청크 수와 가장 긴 청크의 글자 수를 출력하세요.

**확인 기준**: 두 결과 모두 원문 수보다 청크가 많고, 50백분위 쪽 청크가 같거나 많습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안의 순서대로 파서를 만들고, 결과를 Document로 바꿔 원문 위치를 붙입니다.

세부구현:
1. SemanticSplitterNodeParser에 buffer_size, 백분위, index_embedding을 줍니다.
2. to_index_documents로 바꾼 documents를 get_nodes_from_documents에 넘깁니다.
3. 초기 전체 결과를 nodes_to_documents로 바꾼 뒤 restore_positions의 sequential=True로 원문 위치를 붙입니다.
4. 백분위만 바꿔 한 번 더 반복합니다.
```

</details>

In [ ]:
# [제공 코드]
# 이웃 문장의 의미 거리로 경계를 정하는 파서입니다.
from llama_index.core.node_parser import SemanticSplitterNodeParser

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert percentile50_chunks and percentile80_chunks, "두 백분위의 결과를 모두 만드세요."
assert all("window" not in doc.metadata for doc in percentile50_chunks + percentile80_chunks), (
    "SentenceWindowNodeParser가 아니라 SemanticSplitterNodeParser 결과를 담으세요."
)
assert len(percentile80_chunks) > len(records), (
    "80백분위에서도 경계가 생겨 청크가 원문 수보다 많아야 합니다. to_index_documents(documents) 전체를 넣었는지 확인하세요."
)
assert len(percentile50_chunks) >= len(percentile80_chunks), (
    "percentile50_chunks에 50, percentile80_chunks에 80을 넣었는지 확인하세요."
)

for chunks in [percentile50_chunks, percentile80_chunks]:
    assert {doc.metadata["source_id"] for doc in chunks} == set(records_by_id), "documents 전체를 나누세요."
    for doc in chunks:
        meta = doc.metadata
        assert doc.page_content == records_by_id[meta["source_id"]]["text"][meta["start_index"] : meta["end_index"]], (
            "nodes_to_documents → restore_positions로 원문 위치를 붙이세요."
        )

## 3. Parent-Child 검색기 연결하기

**배경**: 작은 자식으로 찾고 부모 원문 전체를 돌려줍니다.

**요구사항**:

- **`pc_retriever`** (`ParentDocumentRetriever`): 자식은 Chroma `pc_store`(컬렉션 `day46_lv2_parent`), 부모는 `InMemoryStore` `pc_docstore`. 부모 `documents`(ID는 `source_id`), 자식 120자·겹침 20, `id_key="parent_id"`, k=3.
- **`child_hits`** (`Document` 리스트): `pc_store`에서 `question_text`로 k=3 검색한 자식.
- **`parent_hits`** (`Document` 리스트): `pc_retriever`로 같은 질문을 검색한 부모. 둘 다 출력하세요.

**확인 기준**: 부모는 1~3개, 자식이 가리킨 부모와 같고 원문 전체입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 자식은 벡터 저장소에, 부모는 docstore에 두고 부모 ID로 잇습니다.

세부구현:
1. Chroma를 만들고 reset_collection으로 비웁니다.
2. child_splitter(RecursiveCharacterTextSplitter)와 InMemoryStore로 ParentDocumentRetriever를 만듭니다.
3. add_documents의 ids에 source_id 목록을 줍니다.
4. 자식은 pc_store.similarity_search, 부모는 pc_retriever.invoke로 얻습니다.
```

</details>

In [ ]:
# [제공 코드]
# 검색용 자식은 Chroma에, 반환할 부모는 InMemoryStore에 보관합니다.
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert pc_retriever.id_key == "parent_id" and pc_retriever.search_kwargs == {"k": 3}, (
    'pc_retriever에 id_key="parent_id"와 search_kwargs k=3을 주세요.'
)

child_texts = pc_store.get()["documents"]
expected_children = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20).split_documents(documents)

assert max(len(text) for text in child_texts) <= 120 and len(child_texts) == len(expected_children), (
    "자식은 120자·겹침 20으로 나눕니다. 다시 실행했다면 reset_collection()으로 컬렉션을 먼저 비우세요."
)
assert 0 < len(parent_hits) <= 3, "parent_hits에는 pc_retriever.invoke 결과(부모 1~3개)를 담으세요."
assert {doc.metadata["source_id"] for doc in parent_hits} == {doc.metadata["parent_id"] for doc in child_hits}, (
    "child_hits가 가리킨 부모와 parent_hits가 같아야 합니다. 같은 question_text로 검색하세요."
)
assert all(doc.page_content == records_by_id[doc.metadata["source_id"]]["text"] for doc in parent_hits), (
    "부모는 원문 전체여야 합니다. add_documents의 ids에 source_id 목록을 주세요."
)

## 4. Window는 검색 뒤에 붙이기

**배경**: 같은 검색 문장에 앞뒤 문장을 붙이면 반환량이 달라집니다.

**요구사항**:

- **`sentence_documents`** (`Document` 리스트): `window_size=1`인 `SentenceWindowNodeParser`의 전체 결과를 `nodes_to_documents`로 바꾼 뒤 `restore_positions`의 `sequential=True`로 위치를 붙인 문장 목록.
- **`sentence_store`** (Chroma, `day46_lv2_sentences`): `sentence_documents` 적재.
- **`sentence_hits`**: `question_text`로 k=3 검색한 결과.
- **`window0`**: `sentence_hits` 그대로 `restore_positions`를 거친 목록.
- **`window1`**: hit마다 `metadata["window"]`를 본문으로 바꾸고 `restore_positions`를 거친 목록.
- **`chars0`**, **`chars1`** (정수): 두 목록의 글자 수 합. 출력하세요.

**확인 기준**: `chars1`이 `chars0`보다 큽니다. 검색은 한 번만 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문장으로 한 번 검색하고, 같은 결과의 본문만 window로 바꿉니다.

세부구현:
1. SentenceWindowNodeParser 전체 결과를 nodes_to_documents로 바꾸고 restore_positions의 sequential=True로 위치를 붙여 적재합니다.
2. similarity_search는 한 번만 호출합니다.
3. 검색 결과의 window0·window1은 원문 순서가 아니므로 restore_positions의 기본 sequential=False를 유지합니다.
```

</details>

In [ ]:
# [제공 코드]
# 검색할 한 문장과 함께 앞뒤 문맥 window를 기록하는 파서입니다.
from llama_index.core.node_parser import SentenceWindowNodeParser

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all("window" in doc.metadata for doc in sentence_documents), (
    "sentence_documents는 SentenceWindowNodeParser 결과여야 합니다(metadata에 window)."
)
assert len(sentence_store.get()["ids"]) == len(sentence_documents), (
    "sentence_store에 sentence_documents를 한 번씩 적재하세요. 다시 실행했다면 reset_collection()으로 먼저 비우세요."
)
assert 0 < len(sentence_hits) <= 3, "sentence_store에서 question_text로 k=3 검색하세요."
assert chars0 == sum(len(doc.page_content) for doc in window0) and chars1 == sum(
    len(doc.page_content) for doc in window1
), "chars0·chars1은 각 목록의 page_content 글자 수 합입니다."
assert chars1 > chars0, (
    'chars1이 chars0보다 커야 합니다. window0은 문장 그대로, window1은 hit.metadata["window"]를 본문으로 씁니다.'
)

for doc in window0 + window1:
    meta = doc.metadata
    assert doc.page_content == records_by_id[meta["source_id"]]["text"][meta["start_index"] : meta["end_index"]], (
        "restore_positions로 원문 위치를 붙이세요."
    )

## 5. Auto-merging으로 계층 검색하기

**배경**: 토큰 크기로 만든 자식이 기준을 넘게 검색되면 부모로 반환합니다.

**요구사항**:

- **`auto_retriever`** (`AutoMergingRetriever`): `documents`를 `HierarchicalNodeParser`(토큰 `[256, 64]`, 겹침 0, `include_prev_next_rel=False`)로 나누고, 전체 노드는 docstore, 잎만 `VectorStoreIndex`(`index_embedding`)에 넣습니다. 잎 검색 k=3, `simple_ratio_thresh=0.5`.
- **`auto_nodes`**: 계층 파서가 만든 전체 노드(부모와 잎).
- **`auto_hits`**: `question_text`로 검색한 결과.
- **`auto_contexts`** (`Document` 리스트): `auto_hits`의 `.node`를 `nodes_to_documents` → `restore_positions`로 바꾼 것. `nodes_to_documents`의 두 번째 인자로 전체 계층 `auto_nodes`, 세 번째 인자로 청킹 전 원문 `records_by_id`를 전달합니다. 제목·글자 수·본문을 출력하세요.

**확인 기준**: 반환 위치가 원문과 맞습니다. 이 질문에서는 병합이 일어나지 않을 수도 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 부모도 docstore에 두지만 임베딩할 검색 후보는 잎뿐입니다.

세부구현:
1. HierarchicalNodeParser.from_defaults로 계층 노드를 만듭니다.
2. StorageContext의 docstore에 전체 노드를, VectorStoreIndex에 get_leaf_nodes 결과만 넣습니다.
3. 인덱스의 as_retriever와 storage를 AutoMergingRetriever에 넘깁니다.
4. nodes_to_documents에 검색 노드, auto_nodes, records_by_id를 전달하고 restore_positions로 원문 위치를 붙입니다.
```

</details>

In [ ]:
# [제공 코드]
# 계층을 나누고 잎을 검색한 뒤, 조건을 만족하면 부모로 병합하는 도구입니다.
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.retrievers import AutoMergingRetriever

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(auto_retriever, AutoMergingRetriever), "auto_retriever는 AutoMergingRetriever여야 합니다."

root_count = sum(1 for node in auto_nodes if node.parent_node is None)

assert len(get_leaf_nodes(auto_nodes)) > root_count, (
    "잎이 부모보다 많아야 합니다. chunk_sizes는 큰 크기부터 [256, 64]로 적습니다."
)
assert auto_hits and auto_contexts, "auto_retriever.retrieve로 검색하고 Document로 바꾸세요."
assert all("window" not in doc.metadata for doc in auto_contexts), (
    "4번 문장 인덱스가 아니라 계층 잎 인덱스로 검색하세요."
)

for doc in auto_contexts:
    meta = doc.metadata
    assert doc.page_content == records_by_id[meta["source_id"]]["text"][meta["start_index"] : meta["end_index"]], (
        "nodes_to_documents → restore_positions로 원문 위치를 붙이세요."
    )

#### 제공 코드: 근거 구간 Recall

In [ ]:
# [제공 코드]
def evidence_recall(contexts, evidence):
    """반환한 원문 구간들이 고정 근거를 얼마나 회수했는지 계산합니다.

    Args:
        contexts: source_id, start_index, end_index를 가진 원문 Document 리스트.
        evidence: doc_id, start, end를 가진 비어 있지 않은 근거 리스트.
    Returns:
        전체 위치가 회수된 근거 수 / 전체 근거 수. 답변 정확도는 아닙니다.
    """
    covered = {}
    for context in contexts:
        meta = context.metadata
        positions = covered.setdefault(meta["source_id"], set())
        # 출처별 문자 위치의 합집합이므로 겹친 청크가 근거 수를 늘리지 않습니다.
        positions.update(range(meta["start_index"], meta["end_index"]))

    found = 0
    for item in evidence:
        required = set(range(item["start"], item["end"]))
        # 다른 원문의 같은 위치는 인정하지 않으며, 근거 일부만 덮은 경우도 제외합니다.
        if required.issubset(covered.get(item["doc_id"], set())):
            found += 1

    return found / len(evidence)

## 6. 질문별 근거와 반환량 비교하기

**배경**: 한 질문의 결과만으로 전략을 고르지 않습니다.

**요구사항**:

- **`comparison_rows`** (딕셔너리 리스트): 질문마다 3번 `pc_retriever`와 4번 `sentence_store`를 그대로 써서 두 전략을 평가한 행. 키는 `question_id`, `strategy`(`"Parent-Child"` 또는 `"Sentence Window"`), `recall`(`evidence_recall(반환 문서, 질문의 evidence)`), `context_chars`(반환 글자 수 합).
- Sentence Window는 k=3으로 검색한 뒤 window로 바꾼 문맥을 평가합니다. 질문별 표와 전략별 평균 표를 출력하고 아래 서술 칸을 채우세요.

**확인 기준**: 행은 60개(질문 30 × 전략 2)입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문마다 두 전략의 반환 문서를 만들고 같은 함수로 평가합니다.

세부구현:
1. questions를 돌며 Parent-Child는 pc_retriever.invoke, Window는 sentence_store 검색 뒤 window로 바꿉니다.
2. 전략마다 evidence_recall과 글자 수 합을 계산해 행 하나를 추가합니다.
3. 행 목록을 DataFrame으로 만들고 strategy로 묶어 평균을 냅니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(comparison_rows) == len(questions) * 2, "질문마다 두 전략의 행을 하나씩 만드세요."
assert all(set(row) == {"question_id", "strategy", "recall", "context_chars"} for row in comparison_rows), (
    "각 행의 키는 question_id, strategy, recall, context_chars 네 개입니다."
)
assert {(row["question_id"], row["strategy"]) for row in comparison_rows} == {
    (item["question_id"], strategy) for item in questions for strategy in ["Parent-Child", "Sentence Window"]
}, "질문·전략 조합이 빠지거나 겹치지 않아야 합니다. strategy 이름의 철자를 확인하세요."

check_question = questions[0]
check_parents = pc_retriever.invoke(check_question["question"])
check_row = next(
    row
    for row in comparison_rows
    if row["question_id"] == check_question["question_id"] and row["strategy"] == "Parent-Child"
)

assert check_row["context_chars"] == sum(len(doc.page_content) for doc in check_parents), (
    "context_chars는 반환 문서 page_content 글자 수의 합입니다."
)
assert check_row["recall"] == evidence_recall(check_parents, check_question["evidence"]), (
    "recall은 반환 문서와 그 질문의 evidence로 evidence_recall을 계산하세요."
)

*(비교할 question_id와 두 전략의 Recall·반환 글자 수를 적고, 같거나 다른 점을 2문장으로 설명하세요.)*

#### 제공 코드: 군집 요약

In [ ]:
# [제공 코드]
def summarize_groups(groups, level, summary_chain):
    """군집마다 GPT 요약을 한 번 요청해 요약 노드 리스트를 돌려줍니다."""
    summaries = []
    for group_id, members in sorted(groups.items()):
        sections = []
        for node in members:
            # 잎에는 원문 제목을 붙여 서로 다른 문서의 조건이 섞이지 않게 합니다. 요약 노드에는 제목이 없습니다.
            title = node["source"]["title"] if node["level"] == 0 else "하위 요약"
            sections.append(f"[{node['node_id']}] {title}\n{node['text']}")

        summary = summary_chain.invoke({"context": "\n\n".join(sections)})

        # 출처는 GPT에게 묻지 않고 입력 노드에서 모읍니다.
        # 이 실습에서는 노드가 한 군집에만 속해 leaf ID가 겹치지 않습니다. 청크 본문은 겹칠 수 있습니다.
        leaf_ids = [leaf_id for node in members for leaf_id in node["leaf_ids"]]

        # child_ids는 바로 아래 노드, leaf_ids는 계층을 내려갔을 때 도달하는 원문 잎입니다.
        summaries.append({
            "node_id": f"summary_{level}_{group_id}",
            "level": level,
            "text": summary,
            "child_ids": [node["node_id"] for node in members],
            "leaf_ids": leaf_ids,
        })

    return summaries

## 7. 문장 잎에서 첫 요약 계층 만들기

**배경**: 비슷한 문장끼리 묶어 요약하면 넓은 질문을 요약 표현으로 찾을 수 있습니다.

**요구사항**:

- **`leaves`** (딕셔너리 리스트): 4번 `sentence_documents`의 문장마다 잎 하나(형식은 아래 표).
- **`leaf_vectors`**: 잎 `text`를 순서대로 `embedding_model`로 임베딩한 값.
- **`groups`** (`군집 번호 → 잎 리스트`): 정규화한 벡터에 KMeans(3군집, `random_state=0`, `n_init=10`).
- **`summary_chain`**: `{context}` 자리 하나를 받는 요약 프롬프트 → `llm` → `StrOutputParser`(`summarize_groups`가 군집 원문을 `context`로 넘깁니다).
- **`level1`**: `summarize_groups`로 만든 L1 요약 리스트.
- **`leaves_by_id`** (`node_id → 잎`): 요약마다 연결된 잎의 제목·본문을 출력하세요(8번에서 씁니다).

| 잎의 키 | 값 |
|---|---|
| `node_id` | `leaf_0`, `leaf_1`, … 순서대로 |
| `text` | 문장의 `page_content`(window 아님) |
| `level`, `child_ids`, `leaf_ids` | `0`, `[]`, `[node_id]` |
| `source` | 문장의 metadata 복사본 |

**확인 기준**: 요약 3개, 잎은 저마다 한 요약에만 들어갑니다. GPT 요약은 3번 호출됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 잎 만들기·임베딩·군집 뒤에 요약을 한 번씩 요청합니다.

세부구현:
1. sentence_documents를 돌며 표 형식의 잎 딕셔너리를 만듭니다.
2. embed_documents로 잎 text를 임베딩하고, normalize한 벡터에 KMeans를 적용합니다.
3. 군집 번호로 잎을 묶어 groups를 만듭니다(pandas groupby).
4. 요약 체인을 만들고 summarize_groups를 호출합니다.
```

</details>

In [ ]:
# [제공 코드]
# 비슷한 잎을 군집으로 묶고, 군집 원문을 요약 문자열로 받는 도구입니다.
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(set(leaf) == {"node_id", "level", "text", "child_ids", "leaf_ids", "source"} for leaf in leaves), (
    "잎 노드의 키는 표의 여섯 개입니다."
)
assert [leaf["node_id"] for leaf in leaves] == [f"leaf_{index}" for index in range(len(leaves))], (
    "node_id는 순서대로 leaf_0, leaf_1, … 입니다."
)
assert all(
    leaf["level"] == 0 and leaf["child_ids"] == [] and leaf["leaf_ids"] == [leaf["node_id"]] for leaf in leaves
), "잎은 level 0, child_ids 빈 리스트, leaf_ids는 자기 ID 하나입니다."
assert [leaf["text"] for leaf in leaves] == [doc.page_content for doc in sentence_documents], (
    "잎 text는 sentence_documents의 page_content(문장)입니다. window를 쓰지 않습니다."
)
assert len(level1) == len(groups) == 3, "3군집이면 요약도 3개입니다."
assert all(node["level"] == 1 and node["text"].strip() for node in level1), (
    "level1은 summarize_groups(groups, 1, summary_chain)의 결과입니다."
)
assert sorted(child_id for node in level1 for child_id in node["child_ids"]) == sorted(
    leaf["node_id"] for leaf in leaves
), "잎은 저마다 정확히 한 요약에 들어가야 합니다."
assert leaves_by_id == {leaf["node_id"]: leaf for leaf in leaves}, "leaves_by_id는 node_id → 잎 딕셔너리입니다."

## 8. 요약 주장과 생략을 나누어 확인하기

**배경**: 출처가 연결되어 있어도 원문 내용이 모두 보존되지는 않습니다.

**요구사항**:

7번의 `level1[0]`과 `leaves_by_id`로 다음을 기록하세요.
- **주장 확인**: 요약의 실제 주장 1~2개와 연결된 원문 인용을 비교해 지지·버전 혼합·확인 불가로 판단합니다.
- **생략 확인**: 연결 원문의 조건·수치·변경 내용 하나를 골라 요약에 유지됐는지 확인합니다.
- **다음 계층**: L2를 만들 때 임베딩할 문자열을 설명합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 요약에서 원문으로, 원문에서 요약으로 각각 확인합니다.
```

</details>

*(여기에 적으세요: 요약 주장 1개와 원문 인용 → 지지/버전 혼합/확인 불가, 원문 조건·수치 1개 → 유지/생략, L2에서 임베딩할 문자열)*